In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

In [ ]:
files = reader.read()

In [ ]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [ ]:
len(documents)

In [ ]:
from minsearch import Index
index = Index(text_fields = ["content"],
              keyword_fields = ["filename"]
             )
index.fit(documents)

In [ ]:
question = "How does the agentic loop keep calling the model until it stops?"
search_results = index.search(question, num_results = 5)
search_results

In [ ]:
##RAG

In [90]:
from dotenv import load_dotenv
load_dotenv()

from importlib import reload
import ingest
import rag_helper
reload(ingest)
reload(rag_helper)

from ingest import load_lesson_data, build_index
from rag_helper import RAGBase, AgenticRAG


In [91]:
documents = load_lesson_data() 
#documents_chunked = load_lesson_data()

In [92]:
index = build_index(documents)

assistant = RAGBase(
    index=index,
    llm_client = OpenAI()
)

result = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(result.answer)
print(result.usage.input_tokens)


The loop keeps calling the model with a `while True` loop. After each response, it checks whether the model returned any `function_call` items:

- if there are function calls, it runs the tools, appends the tool outputs to `messages`, and calls the model again
- if there are no function calls, it `break`s out of the loop

So the stop condition is simple: **no function calls in the response means the model is done**.
7111


In [82]:
index = build_index(documents)

assistant = AgenticRAG(
    index=index,
)

result = await assistant.rag("How does the agentic loop work, and how is it different from plain RAG?")
print(result.answer)
print(result.usage.input_tokens)
print(result.search_calls)

The search tool was called 3 times.
The core idea of the **agentic loop** is:

1. Send the user question plus instructions to the LLM.
2. The LLM may return a **tool/function call** instead of a final answer.
3. Your code executes the tool and appends the result to the message history.
4. Send the updated history back to the LLM.
5. Repeat until the model returns a final answer with **no more tool calls**.

So the model is effectively “driving” the next step, while your code just keeps the loop going. The lesson describes it as a `while True` loop that:
- calls the model,
- runs any function calls it asks for,
- adds tool outputs back into memory,
- stops when there are no more function calls.

Why it matters: the model can **search multiple times**, **retry with different keywords**, or **recover from a bad first search**. For example, if the query has a typo like “Olama,” the agent can search, see poor results, then search again with “Ollama.”

### How that differs from plain RAG

**

/Users/mo/Developer/llm-zoomcamp-2026-code/01-agentic-rag/homework/rag_helper.py:164: PydanticAIDeprecationWarning: `AgentRunResult.usage` is no longer a method; access it as a property (drop the parentheses).
  # 3. Access the total count via call_tracker["count"]


In [ ]:
#chunks

In [93]:

chunks = chunk_documents(documents, size=2000, step=1000)

In [94]:
len(chunks)

295

In [ ]:
index = build_index(documents_chunked)

assistant = AgenticRAG(
    index=index,
)

result = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(result.answer)
print(result.usage.input_tokens)

In [ ]:
2293/7111

In [ ]:
(7111 - 2293) /(7111)